# 03 — Match Filtered NIST MOFs to CSD Structures

**Purpose:** Link filtered NIST isotherm adsorbents to CSD crystal structure identifiers using DOI-based matching with a full-CSD fallback scan.

**Requires:** CSD Python API licence — run in the CSD Python kernel.

**Inputs:**
- `data/mof_adsorbents_for_cif_matching.json` — filtered NIST MOFs (from notebook 01)
- `data/step-02.csv` — curated CSD MOF subset metadata (from notebook 02)

**Outputs:**
- `data/matched_mofs.csv` — all NIST→CSD matches with metadata and source tracking
- `data/unmatched_mofs.csv` — NIST MOFs that could not be matched to any CSD entry

**Matching pipeline:**
1. DOI matching against CSD MOF subset (~55k unique DOIs)
2. Full CSD fallback scan (~1.4M entries) for remaining unmatched DOIs — uses a pickle cache so subsequent runs are instant
3. Combine matches and export

## 1. Load Data

In [1]:
import pandas as pd
import json
import pickle
from tqdm import tqdm

with open('data/mof_adsorbents_for_cif_matching.json', 'r', encoding='utf-8') as f:
    nist_mofs = json.load(f)

csd = pd.read_csv('data/step-02.csv', encoding='utf-8')

print(f"Filtered NIST MOFs: {len(nist_mofs)}")
print(f"CSD MOF entries:    {len(csd)}")

Filtered NIST MOFs: 418
CSD MOF entries:    135253


## 2. DOI Matching — CSD MOF Subset

Builds bidirectional DOI lookup maps (NIST DOIs → MOF names, CSD DOIs → identifiers) and matches on shared DOIs. Each match records its source as "MOF subset".

In [2]:
# NIST side: DOI → [MOF names]
nist_doi_to_mofs = {}
for mof in nist_mofs:
    for doi in mof.get('DOIs', []):
        nist_doi_to_mofs.setdefault(doi.lower().strip(), []).append(mof['name'])

# CSD MOF subset side: DOI → [identifiers] (excluding dedup-flagged entries)
csd_doi_to_ids = {}
for _, row in csd.iterrows():
    if row['note'] != '-': continue
    doi = row.get('publication_doi')
    if pd.isna(doi): continue
    csd_doi_to_ids.setdefault(str(doi).lower().strip(), []).append(row['identifier'])

shared_dois = set(nist_doi_to_mofs) & set(csd_doi_to_ids)
print(f"NIST unique DOIs:  {len(nist_doi_to_mofs)}")
print(f"CSD unique DOIs:   {len(csd_doi_to_ids)}")
print(f"Shared DOIs:       {len(shared_dois)}")

# Build matches
doi_matches = []
for doi in shared_dois:
    for nist_name in set(nist_doi_to_mofs[doi]):
        hashkey = next((m['hashkey'] for m in nist_mofs if m['name'] == nist_name), None)
        for csd_id in csd_doi_to_ids[doi]:
            doi_matches.append({
                'nist_name': nist_name, 'nist_hashkey': hashkey,
                'csd_identifier': csd_id, 'doi': doi, 'source': 'MOF subset',
            })

df_matched = pd.DataFrame(doi_matches)
matched_subset = set(df_matched['nist_name'].unique()) if len(df_matched) > 0 else set()
print(f"Matched from MOF subset: {len(matched_subset)} unique NIST MOFs ({len(df_matched)} rows)")

NIST unique DOIs:  1127
CSD unique DOIs:   55764
Shared DOIs:       225
Matched from MOF subset: 258 unique NIST MOFs (615 rows)


## 3. Full CSD Fallback Scan (cached)

For NIST MOFs not matched via the MOF subset, scans the entire CSD (~1.4M entries) for matching DOIs. The full scan takes ~25 minutes on the first run; results are cached to `data/full_csd_doi_index.pickle` so subsequent runs load in ~1 s.

In [3]:
import ccdc.io
from pathlib import Path

cache_path = Path("data/full_csd_doi_index.pickle")

# ── Build or load DOI → [identifier] index for the full CSD ─────────────────
if cache_path.exists():
    with open(cache_path, "rb") as f:
        full_csd_doi_index = pickle.load(f)
    print(f"Loaded full CSD DOI index from cache ({len(full_csd_doi_index)} DOIs)")
else:
    print("Building full CSD DOI index (first run only, ~25 min)...")
    reader = ccdc.io.EntryReader('CSD')
    full_csd_doi_index = {}   # doi_lower → [identifier, ...]
    for entry in tqdm(reader, total=len(reader), desc="Scanning full CSD"):
        doi = entry.publication.doi
        if doi is None:
            continue
        full_csd_doi_index.setdefault(doi.lower().strip(), []).append(entry.identifier)
    with open(cache_path, "wb") as f:
        pickle.dump(full_csd_doi_index, f)
    print(f"Saved full CSD DOI index: {len(full_csd_doi_index)} DOIs → {cache_path}")

# ── Match unmatched NIST MOFs against the full CSD index ─────────────────────
unmatched = [m for m in nist_mofs if m['name'] not in matched_subset]
unmatched_dois = set()
unmatched_doi_to_mofs = {}
for mof in unmatched:
    for doi in mof.get('DOIs', []):
        dl = doi.lower().strip()
        unmatched_dois.add(dl)
        unmatched_doi_to_mofs.setdefault(dl, []).append(mof['name'])

print(f"Unmatched NIST MOFs: {len(unmatched)}")
print(f"DOIs to search: {len(unmatched_dois)}")

full_matches = []
for doi in unmatched_dois:
    if doi not in full_csd_doi_index:
        continue
    for csd_id in full_csd_doi_index[doi]:
        for nist_name in set(unmatched_doi_to_mofs[doi]):
            hashkey = next((m['hashkey'] for m in nist_mofs if m['name'] == nist_name), None)
            full_matches.append({
                'nist_name': nist_name, 'nist_hashkey': hashkey,
                'csd_identifier': csd_id, 'doi': doi, 'source': 'full CSD',
            })

df_full = pd.DataFrame(full_matches)
new_names = set(df_full['nist_name'].unique()) - matched_subset if len(df_full) > 0 else set()
print(f"Additional MOFs from full CSD: {len(new_names)} ({len(df_full)} rows)")

Building full CSD DOI index (first run only, ~25 min)...


Scanning full CSD:   2%|▏         | 22403/1413222 [00:27<28:22, 817.02it/s] 


KeyboardInterrupt: 

## 4. Combine & Export Matches

Merges MOF-subset and full-CSD matches, attaches CSD metadata (formula, disorder flag), and saves the combined result.

In [ ]:
df_all = pd.concat([df_matched, df_full], ignore_index=True)

# Attach CSD metadata
csd_meta = csd[['identifier','formula','has_disorder']].drop_duplicates(subset='identifier', keep='first')
csd_meta = csd_meta.rename(columns={'identifier': 'csd_identifier'})
df_all = df_all.merge(csd_meta, on='csd_identifier', how='left')
df_all = df_all.sort_values(['nist_name','csd_identifier']).reset_index(drop=True)

df_all.to_csv('data/matched_mofs.csv', index=False, encoding='utf-8')

total = len(nist_mofs)
matched = df_all['nist_name'].nunique()
print(f"{'='*50}")
print(f"MATCHING SUMMARY (DOI-based)")
print(f"{'='*50}")
print(f"Starting NIST MOFs:          {total}")
print(f"Matched (MOF subset):        {len(matched_subset)}")
print(f"Matched (full CSD fallback): {len(new_names)}")
print(f"Total matched:               {matched} ({100*matched/total:.1f}%)")
print(f"Still unmatched:             {total - matched}")
print(f"Saved: data/matched_mofs.csv ({len(df_all)} rows)")

# Save unmatched
all_matched = set(df_all['nist_name'].unique())
still_unmatched = [m for m in nist_mofs if m['name'] not in all_matched]
pd.DataFrame(still_unmatched).to_csv('data/unmatched_mofs.csv', index=False, encoding='utf-8')
print(f"Saved: data/unmatched_mofs.csv ({len(still_unmatched)} MOFs)")